# 01 · Fetch raw data

**Phase:** data (plan §7.3, steps 1–2). **Looks at outcomes vs predictors?** No.

What this notebook does:
1. Downloads ERT v16 (`ert.csv`), the V-Dem v16 Full+Others file bundled with it (`vdem.RData`) and the ERT codebook, **pinned to one GitHub commit** (`config/project.yaml → sources.ert.commit`).
2. Records every file's SHA-256 in `data/manifests/raw_ert_v16.json` (committed to git).
3. Converts both files to Parquet in `data/interim/`.
4. Re-runs ERT's `get_eps` (in R) on V-Dem truncated at every year 1900–2025, giving each country's episode status **as ERT would have coded it at the time**. This is the real-time state used for the risk sets (plan §3.3).

Runtime: about 1 minute for the downloads and conversion, about 10 minutes for step 4.
Requires: `pip install -e .`; for step 4, R with the packages from `Rscript R/setup.R`.

In [ ]:
from flypredictor import load_config, Paths

cfg = load_config()
paths = Paths.from_root().ensure() 
print("project root:", paths.root)

project root: C:\Users\markg\OneDrive\Desktop\DACCS\DACSS 756 - Machine Learning\DACSS_ML_FP


## 1 · Download at the pinned commit and write the manifest

In [2]:
from flypredictor.fetch import fetch_raw
from flypredictor.manifest import verify_manifest

fetch_raw(cfg, paths)
problems = verify_manifest("raw_ert_v16", paths.manifests, paths.root)
assert problems == [], problems
print("raw files match the manifest")

[get ] https://raw.githubusercontent.com/vdeminstitute/ERT/7fd51ce856a054582820bb0ac0ccae85371ff9be/inst/ert.csv
[get ] https://raw.githubusercontent.com/vdeminstitute/ERT/7fd51ce856a054582820bb0ac0ccae85371ff9be/data/vdem.RData
[get ] https://raw.githubusercontent.com/vdeminstitute/ERT/7fd51ce856a054582820bb0ac0ccae85371ff9be/inst/ERT_codebook_v16.pdf
manifest written: C:\Users\markg\OneDrive\Desktop\DACCS\DACSS 756 - Machine Learning\DACSS_ML_FP\data\manifests\raw_ert_v16.json
raw files match the manifest


## 2 · Convert to Parquet

In [3]:
from flypredictor.fetch import ert_to_parquet, vdem_to_parquet, check_vdem

ert_to_parquet(paths)
vdem_to_parquet(cfg, paths)          # pyreadr if available, otherwise R/convert_vdem.R
check_vdem(paths, expected_last_year=cfg["years"]["last"])

{'rows': 28092,
 'countries': 202,
 'first_year': 1789,
 'last_year': 2025,
 'duplicate_keys': 0}

## 3 · Real-time episode status (ERT re-run on truncated data)

For every year *T*, V-Dem is cut at *T* (with `codingend` capped at *T*) and `get_eps` is re-run with the default parameters. The row for year *T* is the country's episode status as coded in *T*. This fixes two kinds of hindsight in the final ERT coding:
* episodes are only confirmed once the EDI has fallen 0.10, often years after their start year;
* episode end years are only known after the tolerance period has passed.

If R is not available, set `labels.state_source: rule` in `config/project.yaml` and skip this cell; the Python rule agrees with the ERT re-runs on about 99.6% of democracy-years.

In [4]:
import time
from flypredictor.data import ert_jobs

t0 = time.time()
ert_jobs.run_realtime(cfg, paths)          # skips if the output already exists (force=True to rebuild)
print(f"done in {time.time() - t0:.0f} s")
rt = ert_jobs.load_realtime(paths)
rt.groupby("year")["rt_in_dem_episode"].sum().tail(10)

truncated at 1900: 124 country-years, 0 in an autocratization episode (1s)
truncated at 1901: 124 country-years, 0 in an autocratization episode (1s)
truncated at 1902: 125 country-years, 0 in an autocratization episode (1s)
truncated at 1903: 128 country-years, 0 in an autocratization episode (1s)
truncated at 1904: 130 country-years, 0 in an autocratization episode (1s)
truncated at 1905: 130 country-years, 0 in an autocratization episode (1s)
truncated at 1906: 130 country-years, 1 in an autocratization episode (1s)
truncated at 1907: 130 country-years, 2 in an autocratization episode (1s)
truncated at 1908: 130 country-years, 2 in an autocratization episode (1s)
truncated at 1909: 130 country-years, 1 in an autocratization episode (1s)
truncated at 1910: 131 country-years, 1 in an autocratization episode (1s)
truncated at 1911: 134 country-years, 0 in an autocratization episode (1s)
truncated at 1912: 135 country-years, 0 in an autocratization episode (1s)
truncated at 1913: 135 co

year
2016    16
2017    16
2018    16
2019    14
2020    20
2021    22
2022    26
2023    25
2024    26
2025    27
Name: rt_in_dem_episode, dtype: int64

## 4 · Record the derived files in a manifest

In [5]:
from flypredictor.manifest import file_entry, write_manifest

interim = [paths.ert_parquet, paths.vdem_parquet, paths.interim / ert_jobs.REALTIME_FILE]
entries = [file_entry(f, paths.root) for f in interim if f.is_file()]
write_manifest("interim_v16", entries, paths.manifests)
[(e["path"], e["bytes"]) for e in entries]

[('data/interim/ert_v16.parquet', 175074),
 ('data/interim/vdem_v16.parquet', 46204212),
 ('data/interim/ert_realtime_truncated.parquet', 67453)]